# Ticket 319 — 5-feature baseline 재학습

074742 abs cleaned 6-feature pool 에서 `mouse_stop_segment_count` 제외 (NaN imputation 아티팩트 의심) 한 5 feature 로 baseline XGBoost 재학습.

- 데이터: baseline 만 (lv2_human 51 + lv2_macro 101 + balabit 500 = 652)
- split: random_state=42, test_size=0.2, stratified (074742 와 동일)
- model: XGBoost (n_estimators=400, max_depth=3, learning_rate=0.03, random_state=42)
- 산출물: services/ai/train/model_experiment/runs/ticket_319_5feat_<timestamp>/

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import sys
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

try:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    elif hasattr(sys.stdout, "buffer"):
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")
except Exception:
    pass


RANDOM_STATE = 42
TEST_SIZE = 0.2

FEATURES_320 = [
    "time_to_first_click_ms",
    "inter_click_interval_ms",
    "pre_click_mousemove_count",
    "mouse_total_travel_distance_px",
    "mouse_avg_speed_px_per_ms",
    "mouse_max_speed_px_per_ms",
    "mouse_speed_change_mean",
    "mouse_acceleration_mean",
    "mouse_jerk_mean",
    "mouse_path_straightness_score",
    "mouse_path_curvature_mean",
    "mouse_direction_change_count",
    "mouse_overshoot_flag",
    "mouse_hover_dwell_time_ms",
    "mouse_stop_segment_count",
    "mousemove_event_rate",
]

FEATURES_5 = [
    "time_to_first_click_ms",
    "mouse_max_speed_px_per_ms",
    "mouse_jerk_mean",
    "mouse_path_straightness_score",
    "mousemove_event_rate",
]

BASELINE_065547 = {"feature_count": 2, "overall_auc": 1.0, "lv3_auc": 0.96}
ABS_074742 = {"feature_count": 6, "overall_auc": 1.0, "lv3_auc": 0.96}


def source_group(trial_id, label, algorithm_type):
    if 900001 <= trial_id <= 909999:
        return "lv2_human" if label == "human" else "lv2_macro"
    if 910001 <= trial_id <= 910500:
        return "balabit"
    if 930001 <= trial_id <= 930050 or algorithm_type == "lv3_balabit_kde":
        return "lv3_balabit_kde"
    if 940001 <= trial_id <= 949999 or algorithm_type == "lv4_aggressive":
        return "lv4_aggressive"
    return "unknown"


def load_existing_trials(behavior_dir):
    rows = []
    for path in sorted(behavior_dir.glob("trial_*.json")):
        trial = json.loads(path.read_text(encoding="utf-8"))
        trial_id = int(trial.get("trialId") or path.stem.split("_", 1)[1])
        label = trial.get("label")
        metrics = trial.get("metrics") or {}
        algorithm_type = trial.get("algorithm_type")
        row = {
            "trial_id": trial_id,
            "label": label,
            "label_int": 1 if label == "macro" else 0,
            "source_group": source_group(trial_id, label, algorithm_type),
            "algorithm_type": algorithm_type,
        }
        for feature in FEATURES_320:
            row[feature] = metrics.get(feature)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)


def split_frame(df):
    train_idx, test_idx = train_test_split(
        df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label_int"]
    )
    return df.loc[train_idx].copy(), df.loc[test_idx].copy()


def make_xgb():
    return XGBClassifier(
        n_estimators=400,
        max_depth=3,
        learning_rate=0.03,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=1,
        tree_method="hist",
    )


def main():
    try:
        ai_root = Path(__file__).resolve().parents[2]
    except NameError:
        ai_root = Path.cwd().resolve().parents[1]

    parser = argparse.ArgumentParser()
    parser.add_argument("--behavior-dir", type=Path,
                        default=ai_root / "data" / "behavior")
    parser.add_argument("--run-dir", type=Path,
                        default=ai_root / "train" / "model_experiment" / "runs" /
                                f"ticket_319_5feat_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    args = parser.parse_args(args=[])

    args.run_dir.mkdir(parents=True, exist_ok=False)

    existing = load_existing_trials(args.behavior_dir)
    baseline = existing[existing["source_group"].isin({"lv2_human", "lv2_macro", "balabit"})].copy()
    lv3_eval = existing[existing["source_group"].isin({"balabit", "lv3_balabit_kde"})].copy()

    train_df, test_df = split_frame(baseline)

    model = make_xgb()
    model.fit(train_df[FEATURES_5], train_df["label_int"].values)

    overall_pred = model.predict_proba(test_df[FEATURES_5])[:, 1]
    lv3_pred = model.predict_proba(lv3_eval[FEATURES_5])[:, 1]
    overall_auc = float(roc_auc_score(test_df["label_int"].values, overall_pred))
    lv3_auc = float(roc_auc_score(lv3_eval["label_int"].values, lv3_pred))

    metrics = {
        "condition": "5feat",
        "feature_count": len(FEATURES_5),
        "overall_auc": overall_auc,
        "lv3_auc": lv3_auc,
        "features": FEATURES_5,
    }
    (args.run_dir / "metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    feat_imp = {feat: float(val) for feat, val in zip(FEATURES_5, model.feature_importances_)}
    (args.run_dir / "feature_importance.json").write_text(
        json.dumps(feat_imp, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    joblib.dump(model, args.run_dir / "model.joblib")

    overall_str = f"{overall_auc:.4f}"
    lv3_str = f"{lv3_auc:.4f}"
    report = [
        "# Ticket 319 — 5-feature baseline 재학습",
        "",
        "## 비교 표",
        "",
        "| condition          | features | overall | lv3    |",
        "| ------------------ | -------- | ------- | ------ |",
        f"| 065547 baseline    | {BASELINE_065547['feature_count']}        | "
        f"{BASELINE_065547['overall_auc']:.4f}  | {BASELINE_065547['lv3_auc']:.4f} |",
        f"| 074742 abs cleaned | {ABS_074742['feature_count']}        | "
        f"{ABS_074742['overall_auc']:.4f}  | {ABS_074742['lv3_auc']:.4f} |",
        f"| 319 5feat (본 실험) | {len(FEATURES_5)}        | "
        f"{overall_str}  | {lv3_str} |",
        "",
        "## features (5)",
        "",
    ]
    for feat in FEATURES_5:
        report.append(f"- {feat}")
    report += [
        "",
        "## feature importance (XGBoost gain default)",
        "",
        "| feature | importance |",
        "| --- | --- |",
    ]
    for feat, val in sorted(feat_imp.items(), key=lambda x: -x[1]):
        report.append(f"| {feat} | {val:.6f} |")
    report += [
        "",
        "## 파일",
        "",
        "- metrics.json",
        "- feature_importance.json",
        "- model.joblib",
        "- report.md (this)",
    ]
    (args.run_dir / "report.md").write_text(
        "\n".join(report) + "\n", encoding="utf-8"
    )

    print("=== run_dir ===")
    print(args.run_dir)
    print()
    print("=== metrics ===")
    print(f"  overall_auc: {overall_auc:.4f}")
    print(f"  lv3_auc:     {lv3_auc:.4f}")
    print()
    print("=== feature_importance (sorted desc) ===")
    for feat, val in sorted(feat_imp.items(), key=lambda x: -x[1]):
        print(f"  {feat}: {val:.6f}")


if __name__ == "__main__":
    main()